In [ ]:
from openai import OpenAI
from langchain.document_loaders import PyPDFLoader
import pandas as pd
import json
import warnings
import os
import re
import time
import logging
from tqdm import tqdm

from Extraction_Prompts import (
    DEMOGRAPHIC_PROMPT_TEMPLATE,
    CLINICAL_PRESENTATION_PROMPT_TEMPLATE,
    GENOMICS_PROMPT_TEMPLATE,
    DIAGNOSIS_PROMPT_TEMPLATE,
    TREATMENT_PROMPT_TEMPLATE,
    OUTCOME_PROMPT_TEMPLATE
)

warnings.filterwarnings('ignore')

LLM_MODEL = "gpt-5.1"
client = OpenAI(
    base_url=os.getenv("OPENAI_BASE_URL"),  
    api_key=os.getenv("OPENAI_API_KEY")
)

folder_path = './pdfs/'         
output_filename = "ROTD_extraction_results.csv"
max_retries = 5
max_tokens = 10240
temperature = 0.1

FIXED_FIELDS = [
    'pmid', 'pat_id', 'age_y', 'age_m', 'age_d', 'gender', 'ethnicity', 'country', 'past_med_hist',
    'phenotype_names', 'reports',
    'Genetic_test_method', 'zygosity', 'Mutation_type', 'Mutation_gene_name', 'Mutation_hgvs', 'Chromosome',
    'laterality', 'disease', 'location', 'tumor_size', 'stage', 'pathology',
    'treatment',
    'outcome', 'va_pre', 'va_post'
]

In [ ]:
def extract_json_from_response(response):
    try:
        return json.loads(response)
    except json.JSONDecodeError:
        try:
            json_match = re.search(r'\{[\s\S]*\}', response)
            if json_match:
                return json.loads(json_match.group())
        except:
            pass
    return {}

def extract_json_array_from_response(response):
    try:
        return json.loads(response)
    except json.JSONDecodeError:
        try:
            json_match = re.search(r'\[[\s\S]*\]', response)
            if json_match:
                return json.loads(json_match.group())
        except:
            pass
    return []

def fetch_prompts_response(content, max_retries, sleep_duration=1):
    retries = 0
    while retries < max_retries:
        try:
            response = client.chat.completions.create(
                messages=[{"role": "user", "content": content}],
                temperature=temperature,
                model=LLM_MODEL,
                max_tokens=max_tokens
            )
            return response.choices[0].message.content
        except Exception as e:
            logging.error(f"API call failed: {str(e)}")
            retries += 1
            time.sleep(sleep_duration)
            logging.info(f"Retry {retries}/{max_retries}...")
            if retries == max_retries:
                raise Exception("Max retries reached, failing...")
    return None

def load_pdf_text(pdf_path):
    try:
        loader = PyPDFLoader(pdf_path)
        documents = loader.load()
        return "\n".join([doc.page_content for doc in documents])
    except Exception as e:
        logging.error(f"Error loading PDF {pdf_path}: {str(e)}")
        return None

def get_genetic_data(genetic_information):
    genetic_result = {}
    fields = ['Genetic_test_method', 'zygosity', 'Mutation_type', 'Mutation_gene_name', 'Mutation_hgvs', 'Chromosome']
    for field in fields:
        values = []
        for item in genetic_information:
            if isinstance(item, dict):
                val = item.get(field, '')
                if val:
                    values.append(val)
        genetic_result[field] = json.dumps(values, ensure_ascii=False) if values else "[]"
    return genetic_result

In [ ]:
if not os.path.exists(folder_path):
    os.makedirs(folder_path)

all_patient_rows = []
pdf_files = [f for f in os.listdir(folder_path) if f.endswith('.pdf')]

for filename in tqdm(pdf_files, desc="Processing PDFs"):
    pmid = filename.replace('.pdf', '')
    pdf_path = os.path.join(folder_path, filename)
    
    paper_text = load_pdf_text(pdf_path)
    if not paper_text:
        logging.warning(f"Skipping {filename} due to empty text.")
        continue

    # Stage 1: Patient identification
    prompt_demographic = DEMOGRAPHIC_PROMPT_TEMPLATE.format(pmid=pmid, article_text=paper_text)
    response_demographic = fetch_prompts_response(prompt_demographic, max_retries)
    
    try:
        patients = extract_json_array_from_response(response_demographic)
    except Exception as e:
        logging.error(f"Failed to parse demographic JSON for PMID {pmid}: {str(e)}")
        patients = []
    
    logging.info(f"PMID {pmid}: Identified {len(patients)} patients.")

    # Stage 2: Field-specific extraction
    for patient in patients:
        patient_demographics = json.dumps(patient, ensure_ascii=False)
        
        # Clinical Presentation
        prompt_clinical = CLINICAL_PRESENTATION_PROMPT_TEMPLATE.format(
            pmid=pmid, patient_demographics=patient_demographics, article_text=paper_text
        )
        response_clinical = fetch_prompts_response(prompt_clinical, max_retries)
        clinical_data = extract_json_from_response(response_clinical) or {}

        # Genomics
        prompt_genomic = GENOMICS_PROMPT_TEMPLATE.format(
            pmid=pmid, patient_demographics=patient_demographics, article_text=paper_text
        )
        response_genomic = fetch_prompts_response(prompt_genomic, max_retries)
        genomic_data = extract_json_array_from_response(response_genomic)
        genetic_result = get_genetic_data(genomic_data)

        # Diagnosis
        prompt_diagnosis = DIAGNOSIS_PROMPT_TEMPLATE.format(
            pmid=pmid, patient_demographics=patient_demographics, article_text=paper_text
        )
        response_diagnosis = fetch_prompts_response(prompt_diagnosis, max_retries)
        diagnosis_data = extract_json_from_response(response_diagnosis) or {}

        # Treatment
        prompt_treatment = TREATMENT_PROMPT_TEMPLATE.format(
            pmid=pmid, patient_demographics=patient_demographics, article_text=paper_text
        )
        response_treatment = fetch_prompts_response(prompt_treatment, max_retries)
        treatment_data = extract_json_from_response(response_treatment) or {}

        # Outcomes
        prompt_outcome = OUTCOME_PROMPT_TEMPLATE.format(
            pmid=pmid, patient_demographics=patient_demographics, article_text=paper_text
        )
        response_outcome = fetch_prompts_response(prompt_outcome, max_retries)
        outcome_data = extract_json_from_response(response_outcome) or {}

        # Merge row
        row = patient.copy()
        row.update(clinical_data)
        row.update(genetic_result)
        row.update(diagnosis_data)
        row.update(treatment_data)
        row.update(outcome_data)
        all_patient_rows.append(row)
        
        logging.info(f"  - Patient {patient.get('pat_id', 'N/A')}: Extracted.")

    # Checkpoint
    if len(all_patient_rows) % 10 == 0:
        temp_df = pd.DataFrame(all_patient_rows, columns=FIXED_FIELDS)
        temp_df.to_csv("temp_checkpoint.csv", index=False)

In [ ]:
df = pd.DataFrame(all_patient_rows, columns=FIXED_FIELDS)
df.to_csv(output_filename, index=False, encoding='utf-8-sig')

print(f"Extraction completed. Total patients extracted: {len(all_patient_rows)}")
print(f"Results saved to: {output_filename}")

# Note: All automatically extracted results were independently reviewed and corrected by two researchers.
# Discrepancies were resolved through discussion and consensus.